In [1]:
!pip install librosa resampy soundfile
!pip install tensorflow
!pip install "numpy<1.26"

import os
import numpy as np
import librosa
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split




  Using cached numpy-2.0.2-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (19.5 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.25.2
    Uninstalling numpy-1.25.2:
      Successfully uninstalled numpy-1.25.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
daal4py 2021.6.0 requires daal==2021.4.0, which is not installed.
  Using cached numpy-1.25.2-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
daal4py 2021.6.0 requires daal==2021.4.0, which is not installed.
tensorflow 2.20.0 requires num

2025-09-14 00:41:13.108051: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-09-14 00:41:13.144535: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-14 00:41:14.125594: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
!wget -c https://zenodo.org/record/1188976/files/Audio_Speech_Actors_01-24.zip?download=1 -O Audio_Speech_Actors_01-24.zip
!unzip -o -q Audio_Speech_Actors_01-24.zip -d ravdess



--2025-09-14 00:41:15--  https://zenodo.org/record/1188976/files/Audio_Speech_Actors_01-24.zip?download=1
Resolving zenodo.org (zenodo.org)... 188.185.45.92, 188.185.48.194, 188.185.43.25, ...
Connecting to zenodo.org (zenodo.org)|188.185.45.92|:443... connected.
HTTP request sent, awaiting response... 301 MOVED PERMANENTLY
Location: /records/1188976/files/Audio_Speech_Actors_01-24.zip [following]
--2025-09-14 00:41:15--  https://zenodo.org/records/1188976/files/Audio_Speech_Actors_01-24.zip
Reusing existing connection to zenodo.org:443.
HTTP request sent, awaiting response... 416 REQUESTED_RANGE_NOT_SATISFIABLE

    The file is already fully retrieved; nothing to do.



In [3]:
sr_target = 22050  # ensure we have target sample rate

def extract_features(file_path, augment=True):
    try:
        audio, sr = librosa.load(file_path, sr=sr_target)
        if len(audio) < 220:  # skip too-short files
            return None, None, None

        augmented_audios = [audio]

        if augment:
            for n_steps in [-2, 2]:
                # ✅ FIX: pass sr as keyword arg
                augmented_audios.append(librosa.effects.pitch_shift(audio, sr=sr, n_steps=n_steps))
            noise = 0.005 * np.random.randn(len(audio))
            augmented_audios.append(audio + noise)
            shift = int(0.1 * sr)
            augmented_audios.append(np.roll(audio, shift))

        features = []
        for aud in augmented_audios:
            mfcc = librosa.feature.mfcc(y=aud, sr=sr, n_mfcc=n_mfcc)
            if mfcc.shape[1] < 174:
                pad_width = 174 - mfcc.shape[1]
                mfcc = np.pad(mfcc, pad_width=((0, 0), (0, pad_width)), mode='constant')
            elif mfcc.shape[1] > 174:
                mfcc = mfcc[:, :174]
            features.append(mfcc)

        return features, audio, sr
    except Exception as e:
        print(f"Feature extraction failed for {file_path}: {e}")
        return None, None, None


In [4]:
emotions = {"01":"neutral","02":"calm","03":"happy","04":"sad",
            "05":"angry","06":"fear","07":"disgust","08":"surprise"}
n_mfcc = 60
max_len = 174

X, y = [], []
labels = list(emotions.values())  # for converting string labels to integers

# --- Data loading and augmentation ---
for subdir, _, files in os.walk("ravdess/"):
    for file in files:
        if file.endswith(".wav"):
            path = os.path.join(subdir, file)
            try:
                label_str = emotions[file.split("-")[2]]
                
                # extract_features returns a list of augmented features now
                features, audio, sr = extract_features(path, augment=True)  # augment=True adds pitch/noise/time variations
                if features is None:
                    continue

                for feature in features:
                    # Skip if wrong MFCC size
                    if feature.shape[0] != n_mfcc:
                        continue

                    # Pad or truncate time axis
                    if feature.shape[1] < max_len:
                        pad_width = max_len - feature.shape[1]
                        feature = np.pad(feature, ((0,0),(0,pad_width)), mode="constant")
                    elif feature.shape[1] > max_len:
                        feature = feature[:, :max_len]

                    X.append(feature)
                    y.append(labels.index(label_str))  # convert to integer label

            except KeyError:
                pass
            except Exception as e:
                print(f"Error processing {path}: {e}")

# --- Convert to numpy arrays ---
X = np.array(X)
X = X[..., np.newaxis]  # add channel dimension for CNN
y = np.array(y)

print(f"Final dataset size: {X.shape[0]} samples")
print(f"Feature shape (per sample): {X.shape[1:]}")
print(f"Label shape: {y.shape}")

Final dataset size: 7200 samples
Feature shape (per sample): (60, 174, 1)
Label shape: (7200,)


In [5]:
file_path = "ravdess/Actor_01/03-01-01-01-01-01-01.wav"
mfccs, audio, sr = extract_features(file_path, augment=True)
print("Number of features returned:", len(mfccs) if mfccs is not None else 0)
print("MFCC shape of first:", mfccs[0].shape if mfccs is not None else None)


Number of features returned: 5
MFCC shape of first: (60, 174)


In [7]:
X = np.array(X, dtype=np.float32)
y_num = np.array(y, dtype=np.int32)
X = X.reshape(X.shape[0], -1)


In [8]:

from sklearn.preprocessing import StandardScaler

X_reshaped = X.reshape(X.shape[0], -1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_reshaped)
X = X_scaled.reshape(X.shape[0], 60, 174, 1)



In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y_num, test_size=0.2, random_state=42)


In [10]:
import numpy as np
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Reshape X back to (samples, 60, 174, 1)
X = X.reshape(X.shape[0], 60, 174, 1)

model = Sequential()
model.add(Input(shape=(60, 174, 1)))
model.add(Conv2D(32, (3,3), activation='relu', padding='same'))
model.add(MaxPooling2D((2,2)))
model.add(Conv2D(64, (3,3), activation='relu', padding='same'))
model.add(MaxPooling2D((2,2)))
model.add(Flatten())
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(8, activation='softmax'))



E0000 00:00:1757790856.606122   11064 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1757790856.612777   11064 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [11]:

model.compile(loss='sparse_categorical_crossentropy',
              optimizer=Adam(0.0005),
              metrics=['accuracy'])


In [12]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(class_weight='balanced',
                                    classes=np.unique(y_train),
                                    y=y_train)

class_weights = dict(enumerate(class_weights))

history = model.fit(X_train, y_train,
                    validation_data=(X_test, y_test),
                    epochs=50,
                    batch_size=32,
                    class_weight=class_weights)


Epoch 1/50
180/180 ━━━━━━━━━━━━━━━━━━━━ 22s 117ms/step - accuracy: 0.2808 - loss: 2.0942 - val_accuracy: 0.5625 - val_loss: 1.2312
Epoch 2/50
180/180 ━━━━━━━━━━━━━━━━━━━━ 21s 119ms/step - accuracy: 0.6744 - loss: 0.9403 - val_accuracy: 0.7056 - val_loss: 0.7990
Epoch 3/50
180/180 ━━━━━━━━━━━━━━━━━━━━ 21s 118ms/step - accuracy: 0.8630 - loss: 0.4329 - val_accuracy: 0.7736 - val_loss: 0.6421
Epoch 4/50
180/180 ━━━━━━━━━━━━━━━━━━━━ 21s 118ms/step - accuracy: 0.9414 - loss: 0.1943 - val_accuracy: 0.8035 - val_loss: 0.5744
Epoch 5/50
180/180 ━━━━━━━━━━━━━━━━━━━━ 21s 118ms/step - accuracy: 0.9672 - loss: 0.1073 - val_accuracy: 0.8062 - val_loss: 0.6140
Epoch 6/50
180/180 ━━━━━━━━━━━━━━━━━━━━ 22s 121ms/step - accuracy: 0.9790 - loss: 0.0697 - val_accuracy: 0.8097 - val_loss: 0.5837
Epoch 7/50
180/180 ━━━━━━━━━━━━━━━━━━━━ 22s 119ms/step - accuracy: 0.9878 - loss: 0.0451 - val_accuracy: 0.7965 - val_loss: 0.7013
Epoch 8/50
180/180 ━━━━━━━━━━━━━━━━━━━━ 21s 119ms/step - accuracy: 0.9900 - loss: 0

In [13]:
model.save("emotion_model1.h5")
print("✅ Full model trained & saved as emotion_model.h5 in /content/")

✅ Full model trained & saved as emotion_model.h5 in /content/


In [21]:
import IPython.display as ipd

file_path = "ravdess/Actor_01/03-01-01-01-01-01-01.wav"

mfcc_list, audio, sr = extract_features(file_path, augment=False)

print("Playing audio file...")
if audio is not None and sr is not None:
    display(ipd.Audio(audio, rate=sr))
else:
    print("Could not load audio.")

if mfcc_list is not None and len(mfcc_list) > 0:
    # Take the first feature (original audio, not augmented)
    mfcc = mfcc_list[0]

    # Ensure correct shape (60, 174)
    if mfcc.shape[1] < 174:
        pad_width = 174 - mfcc.shape[1]
        mfcc = np.pad(mfcc, ((0,0),(0,pad_width)), mode="constant")
    elif mfcc.shape[1] > 174:
        mfcc = mfcc[:, :174]

    mfcc_input = mfcc.reshape(1, 60, 174, 1)  # batch, height, width, channel
    
    pred_probs = model.predict(mfcc_input)
    pred_idx = np.argmax(pred_probs)
    
    pred_label = list(emotions.values())[pred_idx]  # Use values, not keys
    print("Predicted emotion:", pred_label)
else:
    print("Could not extract features for prediction.")


Playing audio file...


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
Predicted emotion: calm


In [16]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {accuracy*100:.2f}%")

Test Accuracy: 82.78%


In [17]:
from sklearn.metrics import classification_report
y_pred = np.argmax(model.predict(X_test), axis=1)
print(classification_report(y_test, y_pred))


45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step
              precision    recall  f1-score   support

           0       0.77      0.75      0.76        91
           1       0.83      0.85      0.84       180
           2       0.87      0.76      0.81       217
           3       0.70      0.74      0.72       180
           4       0.88      0.88      0.88       182
           5       0.89      0.82      0.85       203
           6       0.79      0.88      0.83       193
           7       0.87      0.90      0.89       194

    accuracy                           0.83      1440
   macro avg       0.82      0.82      0.82      1440
weighted avg       0.83      0.83      0.83      1440



In [23]:
import os
import random
import numpy as np
import IPython.display as ipd

actor_folder = "ravdess/Actor_01"
files = [f for f in os.listdir(actor_folder) if f.endswith(".wav")]
random.shuffle(files)
files_to_check = files[:5]

for file in files_to_check:
    file_path = os.path.join(actor_folder, file)
    features, audio, sr = extract_features(file_path, augment=False)
    
    if features is None:
        print(f"❌ Could not extract features for {file}")
        continue

    mfcc = features[0]
    if mfcc.shape != (60, 174):
        print(f"Skipping {file}, bad shape: {mfcc.shape}")
        continue

    # ✅ Apply same scaling as training
    mfcc_flat = mfcc.reshape(1, -1)
    mfcc_scaled = scaler.transform(mfcc_flat)
    mfcc_input = mfcc_scaled.reshape(1, 60, 174, 1)

    pred_probs = model.predict(mfcc_input, verbose=0)
    pred_idx = np.argmax(pred_probs)
    pred_label = list(emotions.values())[pred_idx]

    print(f"\n🎵 File: {file}")
    print(f"🔊 Predicted Emotion: {pred_label} ({pred_probs[0][pred_idx]*100:.2f}% confidence)")
    display(ipd.Audio(audio, rate=sr))



🎵 File: 03-01-03-02-01-02-01.wav
🔊 Predicted Emotion: happy (100.00% confidence)



🎵 File: 03-01-02-01-02-01-01.wav
🔊 Predicted Emotion: calm (99.99% confidence)



🎵 File: 03-01-07-02-02-02-01.wav
🔊 Predicted Emotion: disgust (100.00% confidence)



🎵 File: 03-01-05-02-01-02-01.wav
🔊 Predicted Emotion: angry (100.00% confidence)



🎵 File: 03-01-05-02-01-01-01.wav
🔊 Predicted Emotion: angry (100.00% confidence)


In [31]:
file_path = "test8.wav"  
features, audio, sr = extract_features(file_path, augment=False)

if features is None:
    print("❌ Could not extract features.")
else:
    mfcc = features[0]
    if mfcc.shape == (60, 174):
        mfcc_flat = mfcc.reshape(1, -1)
        mfcc_scaled = scaler.transform(mfcc_flat)
        mfcc_input = mfcc_scaled.reshape(1, 60, 174, 1)

        pred_probs = model.predict(mfcc_input, verbose=0)
        pred_idx = np.argmax(pred_probs)
        pred_label = list(emotions.values())[pred_idx]

        print(f"🎤 Predicted Emotion: {pred_label}")
        print(f"Confidence: {pred_probs[0][pred_idx]*100:.2f}%")
        import IPython.display as ipd
        display(ipd.Audio(audio, rate=sr))
    else:
        print(f"❌ Wrong MFCC shape: {mfcc.shape}")


🎤 Predicted Emotion: angry
Confidence: 100.00%
